## Casual self attenetion with single head attention
### (Here we consider prev token to predict next)

In [1]:
import torch.nn as nn 
import torch
torch.manual_seed(123)

In [2]:
class Causal_Attention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))
        # register buffer becuase this tensors are not to be trained so these masks will be moved to appropriate devices so that they are not on the same device as trainable model parameters
        
    def forward(self, x):
        
        batch, num_tokens, input_dim = x.shape
        
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        
        attn_scores = queries@keys.transpose(1,2) #Transpose inside the batch therefor the shape index 1,2
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf) #:num_tokens is to handle if any batch has no of tokens less than context-length
        attn_weights = torch.softmax(
            attn_scores/keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights)
        context_vec = attn_weights@values
        
        return context_vec

In [8]:
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

batch = torch.stack((inputs, inputs), dim=0)

context_length = batch.shape[1]

d_in = batch.shape[2]
d_out = batch.shape[2]
dropout=0.2

In [9]:
ca=Causal_Attention(d_in=d_in,d_out=d_out,context_length=context_length,dropout=dropout)
context_vec=ca.forward(batch)
print("##############################INPUT BATCH#####################################")
print(batch.shape)
print(batch)
print("##############################OUTPUT CONTEXT VECTOR BATCH#####################################")

print(context_vec.shape)
print(context_vec)

##############################INPUT BATCH#####################################
torch.Size([2, 6, 3])
tensor([[[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]],

        [[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]]])
##############################OUTPUT CONTEXT VECTOR BATCH#####################################
torch.Size([2, 6, 3])
tensor([[[0.0826, 0.5688, 0.6536],
         [0.3318, 0.5375, 0.8260],
         [0.2257, 0.3650, 0.5613],
         [0.4112, 0.4592, 0.8133],
         [0.4512, 0.3906, 0.7515],
         [0.1187, 0.1297, 0.2109]],

        [[0.0826, 0.5688, 0.6536],
         [0.3318, 0.5375, 0.8260],
         [0.3955, 0.3170, 0.6495],
         [0.3276, 0.3976, 0.6771],
      